# ✈️ AeroLex — Building a Production-Grade RAG Pipeline

A hands-on build of a multi-stage Retrieval-Augmented Generation system over
**airline passenger-rights law** (EU 261/2004, US DOT 14 CFR 260, Delta Contracts
of Carriage).

**The pipeline:**
```
question
  └─▶ condense follow-up            (conversational memory)       [#8]
  └─▶ BM25 lexical  +  Gemini dense  (hybrid retrieval)           [#2,#3,#4]
  └─▶ reciprocal rank fusion         (merge the two)              [#5]
  └─▶ FLAIR feedback re-scoring      (learn from 👍/👎)            [#10]
  └─▶ cross-encoder re-ranking       (precise top-k)              [#6]
  └─▶ grounded answer + [[n]] cites  (+ long-term profile)        [#7,#9]
```

**How to run:** pick the **`Python (aerolex RAG)`** kernel, then run top to bottom.
Everything works offline (extractive answers); with a `GEMINI_API_KEY` you get full
generation. You built each stage? The final section assembles them with the
`aerolex` library.


# Why and when do we use RAG?
# Best practices
# See if we can implement SOTA for our respective sections - if yes, great, if no, try to do a simpler version or talk about it

## 0 · Setup

In [1]:
import time
from pathlib import Path
import numpy as np
from aerolex import config
from aerolex.loaders import load_documents
from aerolex.chunking import chunk_corpus, load_chunks
from aerolex.store import InMemoryVectorStore
from aerolex.embeddings import GeminiEmbedder
from aerolex.bm25 import BM25Index

In [2]:
# Set API key, LLM, Embedding Model 
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
GEN_MODEL = "gemini-3.5-flash-lite"
KEY = config.load_api_key()
CACHE_DIR = r"C:\src\IEEEWorkshop\aerolex-production-rag\cache"

## 1 · Hierarchical Chunking  ·  Feature #1 - Advitya
Tasks:
- Figure out how to pull information from referenced sections/articles
- One idea - Add references key in the json and add chunk ids of referenced articles

Legal text is a **hierarchy** (Article / § / RULE → paragraph). We split on those
*logical* boundaries — never fixed-size windows — and attach citation metadata, so
an answer can cite *"EU 261/2004 — Article 7"* instead of *"chunk 42"*.

In [3]:
docs = load_documents()
for d in docs:
    print(f"{d.doc_id:16s} {d.doc_type:6s} {len(d.text):>7,d} chars  {d.title}")

# Prefer the pre-built cache (instant); otherwise chunk live.
try:
    chunks = load_chunks()
    print(f"\nloaded {len(chunks)} chunks from cache")
except FileNotFoundError:
    chunks = chunk_corpus(docs)
    print(f"\nchunked live: {len(chunks)} chunks")

by_id = {c.chunk_id: c for c in chunks}
art7 = next(c for c in chunks if c.section_label == "Article 7")
print("\nExample chunk —", art7.citation)
print(art7.text[:320], "…")

delta_domestic   delta   78,663 chars  Delta Domestic General Rules Tariff (Contract of Carriage)
delta_intl       delta  118,099 chars  Delta International General Rules Tariff (Contract of Carriage)
eu261            eu      27,544 chars  EU Regulation (EC) No 261/2004
cfr260           cfr     26,287 chars  US DOT 14 CFR Part 260

loaded 234 chunks from cache

Example chunk — EU Regulation (EC) No 261/2004 — Article 7
1.
Where reference is made to this Article, passengers shall
receive compensation amounting to:
(a) EUR 250 for all flights of 1 500 kilometres or less;
(b) EUR 400 for all intra-Community flights of more than
1 500 kilometres, and for all other flights between 1 500
and 3 500 kilometres;
(c) EUR 600 for all flights no …


## Emebdding Creation - Nutan

In [4]:
print(f"{len(chunks)} chunks loaded!")

234 chunks loaded!


In [28]:
if not KEY:
    print("GEMINI_API_KEY not set.")

print(f"Embedding {len(chunks)} chunks with {EMBED_MODEL} "
        f"(dim={EMBED_DIM}) …")

t0 = time.time()
embedder = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM)
matrix = embedder.embed_documents([c.searchable_text for c in chunks])
print(f"Embedded in {time.time() - t0:.1f}s, shape={matrix.shape}")

Embedding 234 chunks with gemini-embedding-001 (dim=768) …
Embedded in 132.4s, shape=(234, 768)


In [ ]:
# create a matrix with id and store using numpy

store = InMemoryVectorStore([c.chunk_id for c in chunks], matrix)
store.save(Path(CACHE_DIR) / "new_embeddings.npz")

In [5]:
# if time out errors load, already created vector store
store = InMemoryVectorStore.load(Path(CACHE_DIR) / "new_embeddings.npz")
print (f"{store.matrix.shape[0]} Embeddings loaded successfully of shape {store.matrix.shape[1]}!")

234 Embeddings loaded successfully of shape 768!


## 2 · Lexical Search with BM25  ·  Feature #2 - Nutan

BM25 (Best Match 25) is a probabilistic ranking algorithm to measure how relevant a document is to a query, combining term frequency, inverse document frequency, and document‑length normalization.
Use `BM25Okapi` directly with the same stopword-filtering tokenizer for chunks and queries. Scores are lexical relevance scores, not cosine similarities.

### Keyword tokenization

In [6]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def tokenize(text: str) -> list[str]:
    # TODO Fetch list of keywords 
    tokens = re.findall(r"[a-z]+|\u00A7|\d+(?:[./-]\d+)*", text.lower())
    return [token for token in tokens if token not in ENGLISH_STOP_WORDS]

assert tokenize("What is the refund for a cancelled flight?") == ["refund", "cancelled", "flight"]
print("tokenize OK")

tokenize OK


In [7]:
# Tokenize the query for BM25 search
query_tokens = tokenize("What is the refund for a cancelled flight")

In [8]:
# Initialize BM25 index with chunk IDs and their corresponding searchable text
bm25 = BM25Index([c.chunk_id for c in chunks], [c.searchable_text for c in chunks])

In [30]:
kw_search_chunks = []
for cid, score in bm25.search(query_tokens, top_k=5):
    print(f"{score:6.2f}  {by_id[cid].citation}")
    kw_search_chunks.append((cid, score))

  5.21  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
  4.72  US DOT 14 CFR Part 260 — § 260.1
  4.67  US DOT 14 CFR Part 260 — § 260.6 (part 6/6)
  4.66  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
  4.47  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)


## 3 · Vector Search - Nutan

Gemini embeds every chunk into a vector; the whole corpus is saved in one NumPy matrix.
Similarity is just a **normalized dot product** (cosine) — no vector database.

### 🛠️ TODO: write `cosine_search` (normalized dot product top-k)

In [12]:
def cosine_search(query_vec, matrix, ids, top_k=5):

    # TODO: 1) L2-normalize query_vec and each row of matrix
    q_hat = query_vec / np.sqrt(np.sum(query_vec ** 2))
    matrix_hat = matrix / np.sqrt(np.sum(matrix ** 2, axis=1, keepdims=True))
    #       2) scores = matrix @ query   (cosine, since normalized)
    scores = matrix_hat @ q_hat
    #       3) return the top_k (id, score) pairs, highest first
    top_k = np.argsort(-scores)[:top_k]
    top_chunks = [(ids[i], scores[i]) for i in top_k]
    return top_chunks

In [13]:
# def cosine_search(query_vec, matrix, ids, top_k=5):
    # TODO: 1) L2-normalize query_vec and each row of matrix
    #       2) scores = matrix @ query   (cosine, since normalized)
    #       3) return the top_k (id, score) pairs, highest first
    # raise NotImplementedError("your turn!")

_hit = cosine_search(store.matrix[0], store.matrix, store.ids, top_k=5)[0]
assert _hit[0] == store.ids[0] and abs(_hit[1] - 1.0) < 1e-4
print("cosine_search OK")

cosine_search OK


In [26]:
# Embed a real query (1 API call) and search. Needs a key; else we reuse a chunk vector.
query = "What is the refund for a cancelled flight?"

query_vector = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM).embed_query(query)


dense_hits = cosine_search(query_vector, store.matrix, store.ids, top_k=5)
vector_search_chunks = []
for cid, s in dense_hits:
    vector_search_chunks.append((cid, s))
    print(f"{s:5.3f}  {by_id[cid].citation}")

0.720  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
0.719  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
0.707  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
0.696  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)
0.694  US DOT 14 CFR Part 260 — § 260.4 (part 1/3)


## 4 · Reciprocal Rank Fusion  ·  Feature #5

BM25 scores and cosine similarity scores live on different scales and produce two different sets of ranked documents. So we fuse by **rank**:
each list contributes `1 / (k + rank)` to an id's score (`k = 60`).

### 🛠️ TODO: write `rrf`

In [28]:
vector_search_chunks

[('delta_intl::rule-23::0', np.float32(0.7197001)),
 ('delta_domestic::rule-22::0', np.float32(0.7194926)),
 ('cfr260::260-6::1', np.float32(0.70701617)),
 ('delta_domestic::rule-22::1', np.float32(0.69601727)),
 ('cfr260::260-4::0', np.float32(0.6936874))]

In [31]:
kw_search_chunks

[('cfr260::260-6::3', 5.209017420297043),
 ('cfr260::260-1::0', 4.716110541876705),
 ('cfr260::260-6::5', 4.665165771246034),
 ('cfr260::260-6::1', 4.655950412277342),
 ('cfr260::260-6::4', 4.467967284438473)]

In [68]:
def rrf(vector_search_chunks, kw_search_chunks, k=60):
    # TODO 
    combined_chunks = [chunk[0] for chunk in kw_search_chunks] + [chunk[0] for chunk in vector_search_chunks]
    from collections import defaultdict
    kw_chunk_rank = defaultdict(int, {chunk[0]: index + 1 for index, chunk in enumerate(kw_search_chunks)})
    vector_chunk_rank = defaultdict(int, {chunk[0]: index + 1 for index, chunk in enumerate(vector_search_chunks)})
    rrf_scores = {}
    for chunk in combined_chunks:
        r1 = kw_chunk_rank[chunk] # rank in bm25
        r2 = vector_chunk_rank[chunk] # rank in vector search
        bm25_term = 1/(r1 + k) if r1 > 0 else 0
        vector_term = 1/(r2 + k) if r2 > 0 else 0
        final_rrf_score = bm25_term + vector_term
        rrf_scores[chunk] = final_rrf_score
    ranked_chunks = sorted(
    rrf_scores.items(), key=lambda item: item[1], reverse=True
)
    return ranked_chunks

In [69]:
ranked_chunks = rrf(kw_search_chunks, vector_search_chunks, k=60)

In [104]:
ranked_chunks

[('cfr260::260-6::1', 0.03149801587301587),
 ('delta_intl::rule-23::0', 0.01639344262295082),
 ('cfr260::260-6::3', 0.01639344262295082),
 ('delta_domestic::rule-22::0', 0.016129032258064516),
 ('cfr260::260-1::0', 0.016129032258064516),
 ('cfr260::260-6::5', 0.015873015873015872),
 ('delta_domestic::rule-22::1', 0.015625),
 ('cfr260::260-4::0', 0.015384615384615385),
 ('cfr260::260-6::4', 0.015384615384615385)]

In [78]:
by_id[ranked_chunks[0][0]]

Chunk(chunk_id='cfr260::260-6::1', doc_id='cfr260', doc_title='US DOT 14 CFR Part 260', doc_type='cfr', section_label='§ 260.6', heading='§ 260.6 Refunding fare for flights cancelled or significantly delayed or changed by carriers.', part_index=1, part_count=6, text="f bag fees if the bag is lost, the right to compensation for\ndamaged, lost, or pilfered bags, or the right to incidental expenses reimbursement arisin\nf bag fees if the bag is lost, the right to compensation for\ndamaged, lost, or pilfered bags, or the right to incidental expenses reimbursement arising from delayed\nbags beyond the agreed upon delivery date, consistent with the Department's regulation in 14 CFR part\n254 and applicable international treaties.\n(a)\nCarriers' obligation to provide refunds —\n(1)\nCarriers' obligation. A covered carrier that is the merchant of record must provide a full and prompt\nrefund of the airfare, including any taxes and ancillary fees, as set forth in paragraph (a)(2) of this\nsect

In [64]:
# More efficient way
# def rrf(rankings, k=60, top_k=10):
#     scores = {}
#     for ranking in rankings:
#         for rank, chunk_id in enumerate(ranking, start=1):
#             scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)
#     return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]

# assert rrf([["b", "a", "c"], ["b", "a", "d"]])[0][0] == "b"
# print("rrf OK")

rrf OK


## 5 · Cross-Encoder Re-Ranking  ·  Feature #6 - TBD

First-stage retrieval scores query and document *independently*. A cross-encoder
reads them **together** (deep cross-attention) and is far more precise — so we run
it only over the small fused pool. (Model is pre-cached; runs locally.)

In [ ]:
from aerolex.rerank import CrossEncoderReranker

candidates = [(cid, by_id[cid].searchable_text) for cid, _ in ranked_chunks]
reranked = CrossEncoderReranker().rerank(query, candidates, top_k=10)
print("Re-ranked top chunks:")
for cid, s in reranked:
    print(f"{s:7.2f}  {by_id[cid].citation}")
# final_chunks = [by_id[cid] for cid, _ in reranked]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Re-ranked top chunks:
  -0.09  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
  -0.40  US DOT 14 CFR Part 260 — § 260.6 (part 6/6)
  -0.51  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
  -0.85  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
  -0.91  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
  -1.69  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
  -2.59  US DOT 14 CFR Part 260 — § 260.4 (part 1/3)
  -2.73  US DOT 14 CFR Part 260 — § 260.1
  -6.04  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)


In [136]:
reranked

[('cfr260::260-6::3', -0.09313550591468811),
 ('cfr260::260-6::5', -0.3972623348236084),
 ('delta_domestic::rule-22::0', -0.5071156024932861),
 ('cfr260::260-6::1', -0.8482917547225952),
 ('cfr260::260-6::4', -0.9069526195526123),
 ('delta_intl::rule-23::0', -1.6935186386108398),
 ('cfr260::260-4::0', -2.591386318206787),
 ('cfr260::260-1::0', -2.734384298324585),
 ('delta_domestic::rule-22::1', -6.039134979248047)]

In [134]:
# final_chunks
print ("\n------------\n".join(
    f"[{c.chunk_id}] ({c.citation})\n{c.text}"
    for c in (final_chunks[:5])
))

[cfr260::260-6::3] (US DOT 14 CFR Part 260 — § 260.6 (part 4/6))
ond to an offer of:
(A)
A significantly delayed or changed flight or an alternative flight and the flight departs
without the consumer; or
page 6 of 9
[DOT-OST-2022-0089, 89 FR 65537, Aug. 12, 2024]
(B)
A voucher, credit, or other form of compensation by the date on which the cancelled flight
was scheduled to depart or the date that the significantly delayed or changed flight
departs.
(b)
Individuals with a disability. A carrier that is the merchant of record must provide a full and prompt refund
to an individual with a disability upon notification by the individual with a disability that he/she does not
want to continue travel because of the significant changes described in paragraphs (b)(1) through (3) of
this section. The covered carrier must also provide a full and prompt refund to any individuals in the same
reservation as the individual with a disability who do not want to continue travel without the individual
with

In [107]:
from google import genai

# ---- build a basic prompt (mirrors build_prompt in generate.py) ----
SYSTEM = (
    "You are AeroLex, an assistant answering questions about airline passenger "
    "rights using ONLY the provided context passages. Every factual sentence "
    "must end with a citation tag like [[1]] or [[2]] referring to the passage "
    "you used. If the context does not answer the question, say so plainly. "
    "Never invent regulations, article numbers, or passage numbers."
)

# number the chunks + attach citations (same as format_context)
context = "\n------------\n".join(
    f"[{c}] ({by_id[c].citation})\n{by_id[c].text}"
    for c, _ in (ranked_chunks[:5])
)

prompt = (
    f"{SYSTEM}\n\n"
    f"Context passages:\n{context}\n\n"
    f"Question: {query}\n\n"
    f"Answer (cite every claim with [[n]]):"
)

# ---- call Gemini directly ----
client = genai.Client()
interaction = client.interactions.create(
    model="gemini-3.5-flash-lite",
    input=prompt,
)
print(interaction.output_text)

For a cancelled flight, a covered carrier that is the merchant of record must provide a full and prompt refund of the airfare, including any taxes and ancillary fees, to a consumer holding a nonrefundable ticket on a scheduled flight to, from, or within the United States if the consumer chooses not to fly on a rebooked alternative flight or accept any voucher, credit, or other form of compensation [[cfr260::260-6::1]]. 

An automatic full refund of the airfare, including taxes and ancillary fees, is also due if a flight is canceled and the consumer is not offered an alternative flight, voucher, credit, or other form of compensation [[cfr260::260-6::1]].

Additionally, under Delta's rules for involuntary refunds due to a flight cancellation:
* If no portion of the ticket has been used, the refund will equal the fare paid [[delta_intl::rule-23::0, delta_domestic::rule-22::0]].
* If a portion of the ticket has been used and termination or interruption occurs at a fare breakpoint, the refu

In [118]:
ranked_chunks

[('cfr260::260-6::1', 0.03149801587301587),
 ('delta_intl::rule-23::0', 0.01639344262295082),
 ('cfr260::260-6::3', 0.01639344262295082),
 ('delta_domestic::rule-22::0', 0.016129032258064516),
 ('cfr260::260-1::0', 0.016129032258064516),
 ('cfr260::260-6::5', 0.015873015873015872),
 ('delta_domestic::rule-22::1', 0.015625),
 ('cfr260::260-4::0', 0.015384615384615385),
 ('cfr260::260-6::4', 0.015384615384615385)]

In [ ]:
print (context)

[cfr260::260-6::1] (US DOT 14 CFR Part 260 — § 260.6 (part 2/6))
f bag fees if the bag is lost, the right to compensation for
damaged, lost, or pilfered bags, or the right to incidental expenses reimbursement arisin
f bag fees if the bag is lost, the right to compensation for
damaged, lost, or pilfered bags, or the right to incidental expenses reimbursement arising from delayed
bags beyond the agreed upon delivery date, consistent with the Department's regulation in 14 CFR part
254 and applicable international treaties.
(a)
Carriers' obligation to provide refunds —
(1)
Carriers' obligation. A covered carrier that is the merchant of record must provide a full and prompt
refund of the airfare, including any taxes and ancillary fees, as set forth in paragraph (a)(2) of this
section to a consumer that holds a nonrefundable ticket on a scheduled flight to, from, or within the
United States for any cancelled flight or significantly delayed or changed flight where the consumer
chooses not to:

## 6 · Grounded Answer with Inline Citations  ·  Feature #7 - Advitya

- What is the state of the art in citations?
- See if we can implement it 

The model answers **only** from numbered passages and tags each claim with `[[n]]`.
A regex post-processor then validates those tags against the real passages and
**drops hallucinated ones** — the difference between "looks cited" and "is cited".

### 🛠️ TODO: write `resolve_citations`

In [ ]:
CITE = re.compile(r"\[\[(\d+)\]\]")

def resolve_citations(answer, index):
    # TODO: keep only [[n]] whose n is a key in `index`; delete the rest.
    # Return (cleaned_answer, ["[[n]] <citation>", ...] for the n's actually used).
    raise NotImplementedError("your turn!")

_c, _r = resolve_citations("A [[1]] and B [[9]].", {1: "EU261 Art 7", 2: "EU261 Art 5"})
assert "[[9]]" not in _c and _r == ["[[1]] EU261 Art 7"]
print("resolve_citations OK")

In [ ]:
from aerolex.generate import Generator

result = Generator().generate(query, final_chunks)
print("OFFLINE" if result["offline"] else "GEMINI", "answer:\n")
print(result["answer"])
print("\nReferences:")
for r in result["references"]:
    print("  ", r)

## Evaluation Strategies - Nutan

In [ ]:
# Retrieval, Answers themselves

## 10 · The Full Pipeline

Run the setup, chunk/index, embedding, tokenizer, cosine-search, and RRF definition cells first. Set `question` below, then run the assembled pipeline: query embedding -> cosine search + BM25 -> RRF -> cross-encoder reranking -> numbered context -> grounded answer and references. It reuses the earlier notebook functions and indexes without memory or profile state. Retrieval searches the whole corpus; source filtering and source-balanced selection are not enabled here.

Embed it using the existing embedder.
Run the earlier cosine_search().
Run bm25.search(tokenize(query)).
Combine rankings using the earlier rrf().
Rerank the fused candidates.
Build and display numbered context passages.
Generate using the existing SYSTEM prompt and Gemini Interactions API.
Validate citation numbers and display the answer and references.

In [120]:
question = "What is the refund for a cancelled flight in EU region?"

In [123]:
from google import genai
from aerolex.rerank import CrossEncoderReranker

query = question
print("Question:", query)

# 1. Use the same embedder as the document vectors.
query_vec = embedder.embed_query(query)

# 2. Semantic retrieval with the notebook's cosine_search.
dense_results = cosine_search(
    query_vec, store.matrix, store.ids, top_k=config.DENSE_TOP_K
)
print(f"\nDense search: {len(dense_results)} candidates")

# 3. Keyword retrieval with the notebook's tokenizer and BM25 index.
sparse_results = bm25.search(tokenize(query), top_k=config.BM25_TOP_K)
print(f"BM25 search: {len(sparse_results)} candidates")

# 4. The notebook's rrf accepts two lists of (chunk_id, score) pairs.
fused_results = rrf(
    dense_results, sparse_results, k=config.RRF_K
)[:config.RRF_TOP_K]
print(f"RRF: {len(fused_results)} candidates")
if not fused_results:
    raise RuntimeError("Retrieval returned no candidates; check the chunks and indexes.")

# 5. Rerank the fused candidates against the original question.
candidates = [(cid, by_id[cid].searchable_text) for cid, _ in fused_results]
ranked = CrossEncoderReranker().rerank(
    query, candidates, top_k=config.RERANK_TOP_K
)
if not ranked:
    raise RuntimeError("Reranking returned no passages for answer generation.")
answer_chunks = [by_id[cid] for cid, _ in ranked]
print("\nReranked passages:")
for i, (cid, score) in enumerate(ranked, start=1):
    print(f"[{i}] {score:.3f}  {by_id[cid].citation}")

# 6. Number the reranked passages to match the SYSTEM citation instructions.
citation_index = {i: chunk.citation for i, chunk in enumerate(answer_chunks, start=1)}
context = "\n------------\n".join(
    f"[{i}] ({chunk.citation})\n{chunk.text}"
    for i, chunk in enumerate(answer_chunks, start=1)
)
print("\nContext passages:\n")
print(context)

# 7. Reuse SYSTEM and the direct Interactions API pattern from above.
prompt = (
    f"{SYSTEM}\n\nContext passages:\n{context}\n\n"
    f"Question: {query}\n\nAnswer (cite every claim with [[n]]):"
)
client = genai.Client(api_key=KEY)
interaction = client.interactions.create(model=GEN_MODEL, input=prompt)
answer = interaction.output_text
if not answer or not answer.strip():
    raise RuntimeError("The model returned an empty answer.")

# 8. Check cited passage numbers and display only the references used.
used_numbers = sorted({int(n) for n in re.findall(r"\[\[(\d+)\]\]", answer)})
invalid_numbers = [n for n in used_numbers if n not in citation_index]
if invalid_numbers:
    raise ValueError(f"The answer cites passages outside the context: {invalid_numbers}")
references = [f"[[{n}]] {citation_index[n]}" for n in used_numbers]
out = {"query": query, "chunks": answer_chunks, "answer": answer, "references": references}
print("\nAnswer:\n")
print(out["answer"])
print("\nReferences:")
for reference in out["references"]:
    print("  ", reference)

Question: What is the refund for a cancelled flight in EU region?

Dense search: 20 candidates
BM25 search: 20 candidates
RRF: 20 candidates


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Reranked passages:
[1] 1.689  EU Regulation (EC) No 261/2004 — Preamble (part 1/6)
[2] -0.093  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
[3] -0.098  US DOT 14 CFR Part 260 — § 260.6 (part 3/6)
[4] -0.277  EU Regulation (EC) No 261/2004 — Preamble (part 4/6)
[5] -0.334  US DOT 14 CFR Part 260 — § 260.6 (part 1/6)

Context passages:

[1] (EU Regulation (EC) No 261/2004 — Preamble (part 1/6))
I
(Acts whose publication is obligatory)
REGULATION (EC) No 261/2004 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 11 February 2004
establishing common rules on compensation and assistance to passengers in the event of denied
boarding and of cancellation or long delay of flights, and repealing Regulation (EEC) No 295/91
(Text with EEA relevance)
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE
EUROPEAN UNION,
Having
regard
to
the
Treaty
establishing
the
European
Community, and in particular Article 80(2) thereof,
Having regard to the proposal from the Commission (1),
Having regard to the opinion 

## 11 · Where to go next — citations, state of the art

What we built is **Level 1** citation: the model self-tags `[[n]]`, and we validate
the *number is real*. State-of-the-art adds **verification**:

- **Level 2** structured output — model returns `{sentence, support_ids, quote}`.
- **Level 3** quote-grounded spans — match the quote back into the source for exact
  offsets; drop unverifiable quotes.
- **Level 4** NLI / entailment (ALCE) — check the passage actually *entails* the
  claim; score citation **precision & recall**.

The retrieval stack (chunking → hybrid → fusion → re-rank) is where most answer
quality is won. 🎓 You built all of it.